In [ ]:
from data_loader import WalmartDataLoader, TwinsDataLoader, LalondeDataLoader, ACSDataLoader, IHDPDataLoader
from specific_experiments.Q1.search_space_utils import get_ate_bins_df
from specific_experiments.Q1.search_space_utils import plot_ate_analysis_interactive
import pandas as pd

In [ ]:
df_lalonde = LalondeDataLoader().load_data()
common_causes_lalonde = df_lalonde.columns.difference(["treatment", "outcome"], sort=False).tolist()

df_twins = TwinsDataLoader().load_data()
common_causes_twins = df_twins.columns.difference(["treatment", "outcome"], sort=False).tolist()

df_acs = ACSDataLoader().load_data()
common_causes_acs = df_acs.columns.difference(["treatment", "outcome"], sort=False).tolist()

df_ihdp = IHDPDataLoader().load_data()
common_causes_ihdp = df_ihdp.columns.difference(["treatment", "outcome"], sort=False).tolist()

df_walmart = WalmartDataLoader().load_data()
common_causes_walmart = df_walmart.columns.difference(["treatment", "outcome"], sort=False).tolist()

In [ ]:
MAX_RUN_TIME_SEC = 18000

In [ ]:
# ate_bins_data_lalonde = get_ate_bins_df(df_lalonde, common_causes_lalonde, MAX_RUN_TIME_SEC, 'lalonde')
# ate_bins_data_lalonde.to_csv("ate_bins_data_lalonde.csv", index=False)

In [ ]:
# ate_bins_data_walmart = get_ate_bins_df(df_walmart, common_causes_walmart, MAX_RUN_TIME_SEC, 'walmart')
# ate_bins_data_walmart.to_csv("ate_bins_data_walmart.csv", index=False)

In [ ]:
# ate_bins_data_acs = get_ate_bins_df(df_acs, common_causes_acs, MAX_RUN_TIME_SEC, 'acs')
# ate_bins_data_acs.to_csv("ate_bins_data_acs.csv", index=False)

In [ ]:
ate_bins_data_ihdp = get_ate_bins_df(df_ihdp, common_causes_ihdp, MAX_RUN_TIME_SEC, 'ihdp')
ate_bins_data_ihdp.to_csv("ate_bins_data_ihdp.csv", index=False)

In [ ]:
ate_bins_data_twins = get_ate_bins_df(df_twins, common_causes_twins, MAX_RUN_TIME_SEC, 'twins')
ate_bins_data_twins.to_csv("ate_bins_data_twins.csv", index=False)

In [ ]:
ate_bins_data_twins

In [ ]:

### LOAD DATA #####

ate_bins_data_ihdp = pd.read_csv("ate_bins_data_ihdp.csv")
ate_bins_data_acs = pd.read_csv("ate_bins_data_acs.csv")
ate_bins_data_lalonde = pd.read_csv("ate_bins_data_lalonde.csv")
ate_bins_data_twins = pd.read_csv("ate_bins_data_twins.csv")
ate_bins_data_walmart = pd.read_csv("ate_bins_data_walmart.csv")

In [ ]:

plot_ate_analysis_interactive(ate_bins_data_lalonde, "Lalonde")
plot_ate_analysis_interactive(ate_bins_data_twins, "twins")
plot_ate_analysis_interactive(ate_bins_data_acs, "ACS")
plot_ate_analysis_interactive(ate_bins_data_ihdp, "IHDP")
plot_ate_analysis_interactive(ate_bins_data_walmart, "Walmart")

In [ ]:
from search_methods.probe_ATE_search import ProbManager
from experiments import whole_df_ops, LEGAL_OPS_BY_TYPE, LEGAL_FILL_BY_TYPE
import ast

def get_prob(df, sequence, trans_dict, op_probs, common_causes):
    sequence = ast.literal_eval(sequence)
    pm = ProbManager(list(trans_dict.keys()),
                                   common_causes, op_probs, None, whole_df_ops, LEGAL_OPS_BY_TYPE,
                                   legal_fill_by_type=LEGAL_FILL_BY_TYPE, col_types=df.attrs.get('col_types', None),
                                   fill_columns=[col for col in df.columns if df[col].isna().any()])
    return pm.get_sequence_probability(sequence)


In [ ]:
from experiments import largest_data_transformations, prob_dict

print(f"ATE MUST BE IN [{0-50}, {0+50}]")
df_cool = ate_bins_data_lalonde[(ate_bins_data_lalonde['max_ate_in_bin'] > (0-50)) & (ate_bins_data_lalonde['min_ate_in_bin'] < (0+50)) ]
df_cool['prob'] = df_cool['best_sequence'].apply(lambda x:  get_prob(df_lalonde, x, largest_data_transformations, prob_dict, common_causes_lalonde))
df_sorted = df_cool.sort_values(by='prob', ascending=False)
df_sorted.head()

In [ ]:
df_sorted.iloc[0]['best_sequence']
# from experiments import largest_data_transformations
# from utils import apply_data_preparations_seq, calculate_ate_linear_regression_lstsq
#
# sequence = (('bin_equal_frequency_2', 'x1'), ('bin_equal_frequency_2', 'x3'), ('bin_equal_width_2', 'x5'), ('isolationForest', 'TABLE'))
# transformed_df = apply_data_preparations_seq(df_ihdp.copy(), sequence, largest_data_transformations)
#
# calculate_ate_linear_regression_lstsq(transformed_df, 'treatment', 'outcome', common_causes_ihdp)
